# 01 · Dataset headroom check (Part 1, step 2)

Zero-shot SmolVLM-256M on 200 validation examples each of **TextVQA, DocVQA, OK-VQA, VQAv2**, at two image resolutions:
- `0` = processor default (largest tiling)
- `1024` = longest edge 1024 px (≤ 4 tiles + global, much shorter sequences)

**Decision rule (fixed before seeing results):** most headroom first, then prefer a task that needs fine visual detail (TextVQA/DocVQA), and it must fit the T4 time budget.

Takes ~15–25 min on a T4, mostly streaming the images. Subsets are cached on Drive, so a re-run is fast.

In [ ]:
# --- setup (run every session) ---
import os
REPO_URL = "https://github.com/KaveeshwaraBandara/en4554-lora-vlm-tool-agent.git"
REPO_DIR = "/content/en4554-lora-vlm-tool-agent"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q -r requirements.txt
from common import gpu_summary, get_out_dir
print(gpu_summary()); print("outputs ->", get_out_dir())

In [ ]:
!python -m part1_vlm_lora.src.headroom --n 200 --longest-edge 0 1024 --batch-size 4

## Look at some predictions
Check the answer format: a low score from the format (e.g. a full sentence) means something different from a low score from content.

In [ ]:
import pandas as pd, json
from common import get_out_dir
p = get_out_dir("part1", "headroom") / "predictions.jsonl"
df = pd.read_json(p, lines=True)
pd.set_option("display.max_colwidth", 60)
for (name, le), g in df.groupby(["dataset", "longest_edge"]):
    print(f"\n=== {name} @ {le}  score={100*g.score.mean():.1f}")
    display(g.sample(6, random_state=0)[["question", "pred", "answers", "score"]])

In [ ]:
# paste this table back into the chat
print(open("part1_vlm_lora/results/headroom_summary.csv").read())